# Walkthrough — Clean Pipeline 56-32 (Rafif): Phase A → B → C + Verdict

**Role:** thin wrapper over `clean-pipeline-56-32/scripts/`. Cells call the scripts, display their artifacts, and interpret them. **Zero duplicated logic** — scripts stay the single source of truth.
Locked protocol: `clean-pipeline-56-32/protocol.json` (tumbling 60/60, 13 channels, chrono 70/30 + 1h embargo, train-only scaling, collapse/NO-WINNER guards).

| Phase | Goal | Cell outputs | Paper relevance |
|---|---|---|---|
| Contract | Prove the machinery on synthetic data (no dataset) | PASS line | Methods credibility |
| A. Windows | 60-s tumbling, gap-aware, coverage-gated windows | manifest + 38.177 valid windows | Bab II temporal design |
| B. Split+Scale | Chrono split, embargo, train-only scalers, train-only EDA | manifests + scaler joblibs + EDA figs | Bab II validation design |
| C. Evaluate | 126-run grid (3 scalers × 2 algos × 7 k × 3 seeds) | verdict_ledger + summary + figs | Bab III robustness |
| Recap | Verdict analysis + MISSING-check + export | verdict table | Appendix |

> **Setup:** raw CSV resolved explicitly below (repo `dataset/`, Kaggle Input, or kagglehub). Phase C takes belasan menit CPU (126 fits incl. GMM) — Contract + A + B run in minutes.

#### Cell: Setup — locate the pipeline, resolve the raw CSV
**What:** finds `clean-pipeline-56-32/scripts` from any working dir, imports the three phase modules, resolves the 357 MB raw CSV (repo → Kaggle Input → kagglehub), and prints the locked protocol.
**Why:** the scripts default `--raw` to the author's laptop path — never rely on it; explicit paths keep runs reproducible anywhere.
**Read this:** printed `CP_ROOT`, `RAW_CSV` (must exist), and the protocol summary.

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

def find_cp_root():
    p = Path.cwd()
    tried = []
    for _ in range(6):
        for rel in ['clean-pipeline-56-32', 'datmin/clean-pipeline-56-32']:
            cand = p / rel
            tried.append(str(cand / 'scripts'))
            if (cand / 'scripts' / 'check_contract.py').is_file():
                return cand
        p = p.parent
    raise FileNotFoundError('clean-pipeline-56-32/scripts not found. Tried:\n' + '\n'.join(tried))
CP_ROOT = find_cp_root()
sys.path.insert(0, str(CP_ROOT / 'scripts'))
import check_contract, prepare_windows, split_scale, cluster_evaluate
print('CP_ROOT:', CP_ROOT)
print('imported from:', Path(check_contract.__file__).parent)

def resolve_raw(csv='56-32 1sec data 27029986.csv'):
    repo = CP_ROOT.parent
    for c in [repo / 'dataset' / csv, Path('/kaggle/input')]:
        if c.is_file():
            return c
        if c.is_dir():
            hits = list(c.rglob(csv))
            if hits:
                return hits[0]
    try:
        import kagglehub
        return Path(kagglehub.dataset_download('faruqmahdison/utah-datmin')) / csv
    except Exception as e:
        raise FileNotFoundError('raw CSV not found (repo/Kaggle/kagglehub): ' + str(e))
RAW_CSV = resolve_raw()
print('RAW_CSV:', RAW_CSV)
proto = json.loads((CP_ROOT / 'protocol.json').read_text(encoding='utf-8'))
print('protocol:', proto['protocol_version'], '| windows:', proto['phase_a']['window_seconds'], 's/', proto['phase_a']['stride_seconds'], 's | split:', proto['phase_b']['split'], '| scalers:', proto['phase_b']['candidate_scalers'], '| models:', proto['phase_c']['models'])

#### Cell: Environment & dependencies (reproducibility record)
**What:** installs `kagglehub` only if missing, then prints interpreter + library versions.
**Why:** a run without recorded versions is not reproducible; this cell is the record.
**Read this:** versions printed below — paste them into the paper's reproducibility note if asked.

In [ ]:
import sys
try:
    import kagglehub
except ImportError:
    print('installing kagglehub ...')
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub[pandas-datasets]'])
import sklearn, pandas, numpy, joblib
print('python:', sys.version.split()[0])
print('sklearn:', sklearn.__version__, '| pandas:', pandas.__version__, '| numpy:', numpy.__version__, '| joblib:', joblib.__version__)

#### Cell: Contract checks (no dataset needed)
**What:** runs `check_contract.main()` — synthetic unit tests for sentinel masking, 54/60 coverage edge, feature math, chunk-invariance, gap reset, tumbling anchors, embargo, membership, scaler-state immutability, and verdict rules.
**Why:** prove the machinery correct BEFORE spending 30 minutes on data. A FAIL here stops everything.
**Read this:** single `PASS` line; any AssertionError names the broken contract.

In [ ]:
check_contract.main()

## Phase A — Tumbling windows (`prepare_windows`)
13 channels (11 raw + Hydraulic_P/Mech_P proxies), SPP row-drop, sentinel masking, gap reset on any non-+1s delta, coverage gate ≥54/60, tumbling emit. Expect: 28 segments, 41.271 candidates → 38.177 valid.

In [ ]:
man_a = prepare_windows.build_outputs(RAW_CSV, CP_ROOT)
print('segments:', man_a['segments'], '| resets:', man_a['segment_resets'])
print('candidate:', man_a['candidate_windows'], '| valid:', man_a['valid_windows'], '| rejected:', man_a['rejected_windows'])
print('runtime_s:', man_a['runtime_seconds'])

In [ ]:
import numpy as np
meta = pd.read_csv(CP_ROOT / 'data/processed/window_metadata.csv')
print('reject reasons:')
print(meta[~meta['valid']]['reject_reason'].value_counts().to_string())
feat = pd.read_csv(CP_ROOT / 'data/processed/window_features.csv')
print('features:', feat.shape, '| finite all:', bool(np.isfinite(feat.drop(columns=['window_id']).to_numpy(dtype='float64')).all()))
display(feat.head(3))
display(meta.head(3))

## Phase B — Chronological split + train-only scaling (`split_scale`)
70% awal Train / embargo 1 jam / 30% akhir Test (counts + gap asserted); Standard/Robust/MinMax fit Train-only with state-freeze check; scaler joblibs saved; EDA strictly train-only.

In [ ]:
man_b = split_scale.build_outputs(CP_ROOT)
print('train:', man_b['train_rows'], '| embargo:', man_b['embargo_rows'], '| test:', man_b['test_rows'])
print('actual embargo_s:', round(man_b['actual_embargo_seconds']))
print('scalers fit_rows:', {k: v['fit_rows'] for k, v in man_b['scalers'].items()})

In [ ]:
m = CP_ROOT / 'outputs/metrics'
display(pd.read_csv(m / 'train_descriptive_stats.csv').head(10))
for fig in ['eda_correlation_heatmap.png', 'eda_scale_disparity_boxplots.png', 'eda_scaler_comparison_distributions.png']:
    display(Image(str(CP_ROOT / 'outputs/figures' / fig)))
print('Note vs our EDA: everything above is TRAIN-ONLY (no future leakage); ours is global/strided.')

## Phase C — 126-run grid (`cluster_evaluate`)
3 scalers × KMeans/GMM × k=2..8 × seeds 42/0/1, fit train-only, evaluated on both splits. Rules: COLLAPSE (<1% cluster), SHARP_TEST_DROP (Sil −0.10 / DBI ×1.5), UNSTABLE (Sil-range/DBI-range/ARI), else ELIGIBLE. Expect belasan menit CPU (GMM heavy).

In [ ]:
man_c = cluster_evaluate.build_outputs(CP_ROOT)
print('runs:', man_c['run_count'], '| verdicts:', man_c['verdict_count'])
print('status:', man_c['status_counts'])
print('runtime_s:', man_c['runtime_seconds'])

In [ ]:
v2 = CP_ROOT / 'outputs/metrics'
ledger = pd.read_csv(v2 / 'verdict_ledger.csv')
print('== verdict counts ==')
print(ledger['verdict'].value_counts().to_string())
print('== reason codes ==')
print(ledger['reason_codes'].value_counts().to_string())
print('== robust/kmeans/k4 (our locked choice under their rules) ==')
display(ledger[(ledger['scaler']=='robust')&(ledger['algorithm']=='kmeans')&(ledger['k']==4)][['verdict','reason_codes','primary_silhouette_train','primary_silhouette_test','primary_dbi_train','primary_dbi_test']])
print('== ELIGIBLE rows ==')
display(ledger[ledger['verdict']=='ELIGIBLE'][['scaler','algorithm','k','primary_silhouette_train','primary_silhouette_test','primary_dbi_train','primary_dbi_test']])

In [ ]:
for fig in ['cluster_silhouette_train_vs_test.png', 'cluster_dbi_train_vs_test.png', 'cluster_kmeans_inertia_elbow.png']:
    display(Image(str(CP_ROOT / 'outputs/figures' / fig)))
print('blue=Train, orange=Test. Watch train→test drops (non-stationarity) and flat-vs-elbow shapes.')

## Recap — what each artifact proves + open team questions
- Contract PASS → machinery trustworthy. Phase A manifest → window attrition accounted. Phase B manifest → split/embargo/scaler discipline. Ledger → only 4/42 ELIGIBLE; robust/kmeans/k4 NO-WINNER(COLLAPSE) vs our gate-PASS winner: same data, different verdict philosophy (collapse-averse vs idle-tolerant).
- Open: (1) 7 sensors vs 13 channels, (2) verdict philosophy for the paper, (3) regime claims need driller-log validation per protocol.
- Run next cell: MISSING-check + export zip.

In [ ]:
need = ['outputs/manifests/phase_a_manifest.json','outputs/manifests/phase_b_manifest.json','outputs/manifests/phase_c_manifest.json','outputs/metrics/verdict_ledger.csv','outputs/metrics/clustering_summary_primary42.csv','outputs/metrics/train_descriptive_stats.csv','outputs/figures/cluster_silhouette_train_vs_test.png','outputs/figures/cluster_dbi_train_vs_test.png','outputs/figures/cluster_kmeans_inertia_elbow.png','outputs/figures/eda_correlation_heatmap.png']
missing=[f for f in need if not (CP_ROOT/f).exists()]
print('MISSING:', missing if missing else 'none — all clean-pipeline artifacts present')
try:
    import subprocess as _sp
    _rev = _sp.run(['git', '-C', str(CP_ROOT.parent), 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
    print('repo HEAD:', _rev if _rev else 'n/a (not a git checkout)')
except Exception as e:
    print('repo HEAD: n/a -', e)

In [ ]:
from pathlib import Path
import shutil, zipfile
OUT = CP_ROOT / 'outputs'
live = Path('/kaggle/working') if Path('/kaggle/working').exists() else OUT
print('OUT dir (absolute):', OUT.resolve())
zp = live / 'clean_pipeline_walkthrough_outputs.zip'
count = 0
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p.suffix in ('.csv', '.json', '.png'):
            z.write(p, p.relative_to(OUT)); count += 1
print('packed:', zp.resolve(), '| files:', count)
if Path('/kaggle/output').exists():
    import shutil as _sh
    pub = Path('/kaggle/output') / 'clean_pipeline_walkthrough'
    pub.mkdir(parents=True, exist_ok=True)
    _sh.copy2(zp, pub / zp.name)
    for extra in ['metrics/verdict_ledger.csv','metrics/clustering_summary_primary42.csv','manifests/phase_a_manifest.json','manifests/phase_b_manifest.json','manifests/phase_c_manifest.json']:
        s = OUT / extra
        if s.exists():
            _sh.copy2(s, pub / Path(extra).name)
    print('mirrored key artifacts to (persisted):', pub.resolve())
try:
    import google.colab
    from google.colab import files
    files.download(str(zp)); print('browser download started.')
except Exception:
    print('no Colab download; zip at:', zp.resolve())